# Model disagreement: T0 versus T7

Final code session, Part A. The escrow design has two decision points: T0
scores an activation at submission, T7 scores it again at hold-close once
the seven-day downstream log exists. Nothing forces the two to agree. The
20-record smoke test in `scripts/smoke_test_service.py` found zero cases of
a T0 release later judged a T7 reject -- too small a sample to say how often
that happens, or what it costs when it does.

This notebook characterises the disagreement rate only. It does not propose
or implement a mitigation.

**Scope.** This project's cross-validation is time-ordered: each model is
trained on the earlier part of a seed's simulated calendar and tested on the
later part. Only that later region -- roughly half of each seed's dataset --
ever gets an honest, non-leaked prediction; the earlier, fit-only rows have
no unbiased prediction anywhere else in this project either, for the same
reason. "Every activation" below means every activation in that temporal
test region, pooled across all four seeds
(`outputs/multiseed_predictions.parquet`), which is the full evaluation
Phase 5 already ran -- nothing here is retrained.

Bands come from the same cost-optimised thresholds the service is built on:
`outputs/band_operating_point.csv` (T7) and
`outputs/band_operating_point_t0.csv` (T0). No threshold or cost parameter
is touched.


In [1]:
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.eval import disagreement

OUT = REPO_ROOT / "outputs"
pd.set_option("display.width", 220)

method = str(pd.read_csv(OUT / "calibration_choice.csv")["chosen_method"].iloc[0])
paired = disagreement.paired_predictions(method)
context = pd.read_parquet(OUT / "multiseed_row_context.parquet")

transition_by_seed = pd.read_csv(OUT / "disagreement_transition_by_seed.csv")
transition_summary = pd.read_csv(OUT / "disagreement_transition_summary.csv")
release_then_reject_by_seed = pd.read_csv(OUT / "disagreement_release_then_reject_by_seed.csv")
release_then_reject_summary = pd.read_csv(OUT / "disagreement_release_then_reject_summary.csv")
reject_then_release_by_seed = pd.read_csv(OUT / "disagreement_reject_then_release_by_seed.csv")
reject_then_release_summary = pd.read_csv(OUT / "disagreement_reject_then_release_summary.csv")

print(f"calibration method: {method}")
print(f"seeds: {sorted(paired['seed'].unique().tolist())}")
print(f"paired test-region rows (both models, same fold structure): {len(paired)}")


calibration method: isotonic
seeds: [7, 42, 101, 2024]
paired test-region rows (both models, same fold structure): 183220


## A1/A2: the full band-transition matrix

Every paired test-region row, assigned a band by each model's own Phase
6/8-chosen thresholds. Rows are the T0 band; columns are the T7 band. Values
are the mean share across the four seeds; `outputs/disagreement_transition_summary.csv`
also carries the seed-level standard deviation and the pooled (all-seeds-combined)
share alongside it.


In [2]:
matrix_mean = transition_summary.pivot(index="t0_band", columns="t7_band", values="share_mean")
matrix_mean = matrix_mean.reindex(index=["auto_release", "review", "reject"],
                                   columns=["auto_release", "review", "reject"])
(matrix_mean * 100).round(3)


t7_band,auto_release,review,reject
t0_band,,,
auto_release,92.644,2.167,0.172
review,0.940,0.686,0.875
reject,0.145,0.109,2.262


Read by row: given T0's decision, where does T7 send the same activation.
The diagonal (`auto_release`/`auto_release`, `review`/`review`, `reject`/`reject`)
is where the two models agree; every off-diagonal cell is a disagreement.


In [3]:
agreement_share = matrix_mean.to_numpy().trace() if hasattr(matrix_mean.to_numpy(), "trace") else None
diagonal_mean = sum(matrix_mean.loc[b, b] for b in ["auto_release", "review", "reject"])
print(f"overall agreement (same band from both models): {diagonal_mean * 100:.3f}% of test-region rows")
print(f"overall disagreement: {(1 - diagonal_mean) * 100:.3f}% of test-region rows")


overall agreement (same band from both models): 95.592% of test-region rows
overall disagreement: 4.408% of test-region rows


## A3/A4: T0 auto-release, T7 reject -- the payout already made

This is the cell that matters financially: by the time T7 runs, at
hold-close, the escrow has already been released against T0's decision. A
later T7 reject identifies a fraud that has already been paid, not one that
can still be stopped.


In [4]:
release_then_reject_summary[[
    "rate_mean", "rate_sd_across_seeds", "rate_pooled",
    "fraud_share_of_flagged_mean", "fraud_share_of_flagged_sd_across_seeds",
    "commission_value_kes_share_mean", "commission_value_kes_per_1000_activations_mean",
    "n_total_pooled", "n_flagged_pooled", "commission_value_kes_flagged_pooled",
]].round(5)


,rate_mean,rate_sd_across_seeds,rate_pooled,fraud_share_of_flagged_mean,fraud_share_of_flagged_sd_across_seeds,commission_value_kes_share_mean,commission_value_kes_per_1000_activations_mean,n_total_pooled,n_flagged_pooled,commission_value_kes_flagged_pooled
0,0.00172,0.0003,0.00172,0.66643,0.05122,0.00168,136.5,183220,316,25011.2


In [5]:
release_then_reject_by_seed.round(5)


,seed,n_total,n_flagged,rate,fraud_share_of_flagged,commission_value_kes_flagged,commission_value_kes_share,commission_value_kes_per_1000_activations
0,7,45733,60,0.00131,0.68333,4668.01,0.00126,102.071
1,42,44819,84,0.00187,0.63095,6636.56,0.00182,148.075
2,101,46383,79,0.00170,0.62025,6187.32,0.00164,133.396
3,2024,46285,93,0.00201,0.73118,7519.33,0.00200,162.457


## A5: the reverse case -- T0 reject, T7 auto-release

An agent refused at submission on evidence the hold window would have
cleared. This is the other side of the same architectural gap: T0 acts on
less information than T7 ever will.


In [6]:
reject_then_release_summary[[
    "rate_mean", "rate_sd_across_seeds", "rate_pooled",
    "fraud_share_of_flagged_mean", "genuine_share_of_flagged_mean",
    "commission_value_kes_share_mean", "commission_value_kes_per_1000_activations_mean",
    "n_total_pooled", "n_flagged_pooled",
]].round(5)


,rate_mean,rate_sd_across_seeds,rate_pooled,fraud_share_of_flagged_mean,genuine_share_of_flagged_mean,commission_value_kes_share_mean,commission_value_kes_per_1000_activations_mean,n_total_pooled,n_flagged_pooled
0,0.00145,0.0006,0.00145,0.26368,0.73632,0.00145,118.225,183220,265


In [7]:
reject_then_release_by_seed.round(5)


,seed,n_total,n_flagged,rate,fraud_share_of_flagged,commission_value_kes_flagged,commission_value_kes_share,commission_value_kes_per_1000_activations,genuine_share_of_flagged
0,7,45733,77,0.00168,0.23377,6045.13,0.00163,132.1830,0.76623
1,42,44819,71,0.00158,0.30986,6313.98,0.00173,140.8770,0.69014
2,101,46383,27,0.00058,0.33333,2151.04,0.00057,46.3755,0.66667
3,2024,46285,90,0.00194,0.17778,7103.11,0.00189,153.4650,0.82222


## A6: table to paste into the document

One row per finding, headline numbers only (mean across the four seeds, with
seed-level spread; the pooled count-weighted figure alongside it since seeds
carry slightly different row counts).


In [8]:
paste_table = pd.DataFrame([
    {
        "finding": "Overall disagreement rate (any band mismatch)",
        "value": f"{(1 - diagonal_mean) * 100:.3f}%",
        "detail": "share of paired test-region rows where T0 and T7 assign a different band",
    },
    {
        "finding": "T0 auto_release -> T7 reject rate",
        "value": f"{release_then_reject_summary['rate_mean'].iloc[0] * 100:.4f}% "
                 f"(sd {release_then_reject_summary['rate_sd_across_seeds'].iloc[0] * 100:.4f} pp)",
        "detail": f"{release_then_reject_summary['n_flagged_pooled'].iloc[0]} of "
                  f"{release_then_reject_summary['n_total_pooled'].iloc[0]} pooled rows",
    },
    {
        "finding": "  ...of which actually fraudulent",
        "value": f"{release_then_reject_summary['fraud_share_of_flagged_mean'].iloc[0] * 100:.2f}%",
        "detail": "share of the flagged rows where label == 1 -- the rest were genuine "
                  "activations T7 wrongly flagged after T0 had already released them",
    },
    {
        "finding": "  ...commission value at stake",
        "value": f"KES {release_then_reject_summary['commission_value_kes_flagged_pooled'].iloc[0]:,.0f} "
                 f"pooled across all seeds",
        "detail": f"{release_then_reject_summary['commission_value_kes_per_1000_activations_mean'].iloc[0]:.1f} "
                  "KES per 1,000 activations scored; "
                  f"{release_then_reject_summary['commission_value_kes_share_mean'].iloc[0] * 100:.3f}% "
                  "of all commission value in the test region",
    },
    {
        "finding": "T0 reject -> T7 auto_release rate",
        "value": f"{reject_then_release_summary['rate_mean'].iloc[0] * 100:.4f}% "
                 f"(sd {reject_then_release_summary['rate_sd_across_seeds'].iloc[0] * 100:.4f} pp)",
        "detail": f"{reject_then_release_summary['n_flagged_pooled'].iloc[0]} of "
                  f"{reject_then_release_summary['n_total_pooled'].iloc[0]} pooled rows",
    },
    {
        "finding": "  ...of which actually genuine (would have cleared)",
        "value": f"{reject_then_release_summary['genuine_share_of_flagged_mean'].iloc[0] * 100:.2f}%",
        "detail": "share of the flagged rows where label == 0",
    },
])
paste_table


,finding,value,detail
0,Overall disagreement rate (any band mismatch),4.408%,share of paired test-region rows where T0 and ...
1,T0 auto_release -> T7 reject rate,0.1725% (sd 0.0302 pp),316 of 183220 pooled rows
2,...of which actually fraudulent,66.64%,share of the flagged rows where label == 1 -- ...
3,...commission value at stake,"KES 25,011 pooled across all seeds","136.5 KES per 1,000 activations scored; 0.168%..."
4,T0 reject -> T7 auto_release rate,0.1449% (sd 0.0597 pp),265 of 183220 pooled rows
5,...of which actually genuine (would have cle...,73.63%,share of the flagged rows where label == 0


## Reading these numbers

- The two models agree on a band for the large majority of the test region;
  disagreement is concentrated in the `review` band, which is expected --
  `review` is the narrowest band for both models, so it is the easiest one
  to fall out of when a second model with different information re-scores
  the same activation.
- The T0-release-then-T7-reject rate is small in share terms, but it is the
  one failure mode where the platform's own money has already moved: every
  row counted there is a payout the T7 model would have refused, made before
  T7 ever saw the case.
- Not every flagged case in that cell is actually fraud -- see the
  fraud-share figure above. A flagged case that turns out genuine is not
  free either: it means T7 revised T0's decision on an activation that had
  already been paid, which is exactly the situation
  `docs/model_card.md`'s new limitation describes and `/explain/{activation_id}`
  is built to surface to a dispute review.
- No mitigation is proposed here. This notebook only establishes the rate
  and the value at stake so that a mitigation, if one is wanted, can be
  scoped against a real number instead of the 20-record smoke test's zero.
